## SWE - JAX

JAX takes a fundamentally different angle from NumPy and Numba. You write
the kernel as a **pure function over whole arrays**, decorate it with
`@jax.jit`, and the XLA compiler produces a single GPU program for the
entire time loop, running every step on the device without returning to Python.

### Table of Contents

1. [Imports and reference setup](#sec1)
2. [The step as a pure function](#sec2)
3. [Fuse the time loop with `lax.scan`](#sec3)
4. [Acceptance gate](#sec4)
5. [Limitation: shape-rigidity](#sec5)

### <a id="sec1"></a>1. Imports and reference setup

We re-import `swe_core` for the float64 reference field, then load JAX.

In [ ]:
import time

import numpy as np
import matplotlib.pyplot as plt

import swe_core

# Shared problem parameters
N       = 16384
L       = 10.0
H0      = 1.0
AMP     = 0.1
SIG     = 0.5
CFL     = 0.4
G       = 9.81
N_STEPS = 1000
dx      = L / N
DT      = swe_core.fixed_dt(H0 + AMP, dx, cfl=CFL, g=G)

# Float64 NumPy reference
def run_numpy_reference():
    h, hu = swe_core.bump_ic(N, L=L, h0=H0, amplitude=AMP, sigma=SIG)
    for _ in range(N_STEPS):
        swe_core.apply_bc_reflective(h, hu)
        h, hu = swe_core.step_numpy(h, hu, dx, DT, g=G)
    return h, hu

h_ref, hu_ref = run_numpy_reference()

import jax
import jax.numpy as jnp
from jax import lax

### <a id="sec2"></a>2. The step as a pure function

Let's look at the same operation shape as notebook 04, expressed in JAX:

- **Read** — `h[:-1]`, `h[1:]` for interface states, same as NumPy-style slicing.
- **Compute** — Rusanov formula unchanged: `F_h = 0.5*(huL+huR) - 0.5*a*(hR-hL)`.
- **Update** — JAX arrays are immutable, so we produce a new array with `h.at[1:-1].set(...)` instead of `h[1:-1] = ...`.

The function signature `(state, _) -> ((new_state), None)` is the shape `lax.scan` expects in the next section.

---

**Quick Docs**

- `jax.jit(fn)`: just-in-time compile a Python function on its first call
  with a given input shape/dtype. Subsequent calls reuse the compiled code.
- `jax.numpy` mirrors NumPy with immutable arrays.
- `jnp.maximum`, `jnp.abs`, `jnp.sqrt`: elementwise math.
- `jnp.array.at[idx].set(value)` functional update for an immutable array, returns a *new* array with the slot set.
- `jax.lax.scan(body, init, xs)`: a structured loop the compiler can fuse.
  Equivalent to a Python `for x in xs: carry, y = body(carry, x)`, but
  inside a single XLA program.
- `arr.block_until_ready()`: synchronise; required before stopping any GPU
  timer.

In [ ]:
@jax.jit
def step_jax(state, _):
    '''One Rusanov-flux step, fully functional. Returns (new_state, None).

    TODO: fill in the functional Rusanov step below.

    The shape of the answer:
      - h, hu = state  (each shape (N+2,))
      - apply reflective BCs via .at[0].set(...) / .at[-1].set(...)
      - compute interface states at i+1/2 for i = 0..N
      - h_safe = max(h, DRY); u = hu / h_safe; c = sqrt(g h_safe)
      - a = max(|uL|+cL, |uR|+cR)
      - F_h  = 0.5(huL+huR) - 0.5 a (hR - hL)
      - F_hu = 0.5(huL*uL + 0.5 g hL^2 + huR*uR + 0.5 g hR^2) - 0.5 a (huR - huL)
      - h_new  = h.at[1:-1].set(h[1:-1]  - (dt/dx)(F_h[1:]  - F_h[:-1]))
      - hu_new = hu.at[1:-1].set(hu[1:-1] - (dt/dx)(F_hu[1:] - F_hu[:-1]))

    Return ((h_new, hu_new), None) for use with lax.scan.
    '''
    h, hu = state
    DRY = jnp.float32(swe_core.DRY_TOL)
    # TODO: implement the Rusanov step (see swe_core.step_numpy for the
    # equivalent NumPy version; the JAX version differs only in functional
    # updates with .at[idx].set(...)).
    h_new = ...
    hu_new = ...
    return (h_new, hu_new), None

### <a id="sec3"></a>3. Fuse the time loop with `lax.scan`

`for _ in range(N_STEPS): state = step_jax(state, ...)` would
*work*, but each call would round-trip through the Python interpreter
between steps, preventing XLA from being able to fuse them.

`jax.lax.scan(body, init, xs)` collapses the entire loop into one device
program. The cost is that `step_jax` must already be jit-able, and
the scan body must accept a `(state, x)` signature.

---

**Quick Docs**

- `jax.lax.scan(body, init, xs)`: walks the leading axis of `xs`,
  threading `state` through `body`. We do not need the per-step `x` value
  (we use a fixed `DT`), so we pass `jnp.arange(N_STEPS)` and ignore it
  inside the body.
- `jax.tree_util.tree_map(lambda a: a.block_until_ready(), pytree)`:
  synchronise every leaf in a pytree of arrays. We use it as the sync
  before stopping a timer.

In [ ]:
def run_jax(N_local: int):
    '''One full N_STEPS simulation at a given grid size, returns (h, hu) on host.

    TODO: assemble the IC on device, then time-step via jax.lax.scan.
      - dx_local = L / N_local
      - h_np, hu_np = swe_core.bump_ic(N_local, L=L, h0=H0, amplitude=AMP, sigma=SIG)
      - state = (jnp.asarray(h_np, dtype=jnp.float32), jnp.asarray(hu_np, ...))
      - final, _ = jax.lax.scan(step_jax, state, jnp.arange(N_STEPS))
      - return host arrays (call jax.block_until_ready on each before returning)

    Hint: a single jax.lax.scan call replaces an entire Python `for` loop and
    lets XLA fuse all N_STEPS Rusanov steps into one device program.
    '''
    dx_local = L / N_local
    state = ...
    final, _ = ...
    return ...

### <a id="sec4"></a>4. Acceptance gate

Cold time captures the first call (XLA compile + execute). Warm time is
the steady-state, after a couple of warmups. We also check the JAX
float32 result against the NumPy float64 reference.

---

**Quick Docs**

- A float32 trajectory accumulates round-off of order `1e-4` over 1000 steps
  (~n·ε); the measured max-diff vs float64 is ~2e-5. The acceptance
  tolerance is `1e-4` here.

In [ ]:
# Cold capture.
t0 = time.perf_counter()
h_f, hu_f = run_jax(N)
cold_s = time.perf_counter() - t0
print(f'cold  N={N} x {N_STEPS} steps: {cold_s*1000:7.1f} ms  (incl. XLA compile)')

# Warm timing.
def _run():
    return run_jax(N)
warm = swe_core.timed_run(
    _run, warmup=2, repeats=5,
    sync_fn=lambda: None,   # run_jax already blocks before returning
    label='02_jax',
)
print(f'warm  N={N} x {N_STEPS} steps: median {warm["median_s"]*1000:7.1f} ms')

# Acceptance.
diff = swe_core.max_diff(h_ref, h_f)
print(f'max_diff(jax_fp32, numpy_fp64) = {diff:.3e}')
TOL = 1e-4
assert diff < TOL, f'FAIL: {diff:.3e} >= {TOL:.0e}'
print(f'PASS: within {TOL:.0e} of the NumPy reference.')

swe_core.save_timing(
    warm, grid_str=f'N={N}', tool='jax', hardware='gpu',
    dtype='float32', steps=N_STEPS,
    cold_s=cold_s, max_diff_vs_numpy=diff,
)

### <a id="sec5"></a>5. Limitation: shape-rigidity

The XLA compile happens **per input shape**. Re-time at three grid
sizes; the cold time on each new `N` is paid in full because the
previous compile is for a different shape.

---

**Quick Docs**

- A fresh `N` triggers a fresh compile. JAX requires the same shape and dtype for a warm run.


In [ ]:
# EXTRA CREDIT: rerun at fresh shapes (none is the notebook's N=16384, which is
# already compiled), capturing cold + warm at each.
shape_data = []
for N_local in [4096, 8192, 32768, 65536]:
    t0 = time.perf_counter()
    run_jax(N_local)
    cold = time.perf_counter() - t0
    # Warm at this shape.
    warm_med = swe_core.timed_run(lambda Nv=N_local: run_jax(Nv),
                                  warmup=2, repeats=3,
                                  sync_fn=lambda: None,
                                  label=f'02_jax_N{N_local}')['median_s']
    shape_data.append((N_local, cold, warm_med))
    print(f'  N={N_local:>5}:  cold {cold*1000:7.1f} ms   warm {warm_med*1000:7.1f} ms')

# Plot: cold dominates each new shape; warm scales ~linearly with N.
Ns       = [d[0] for d in shape_data]
cold_ms  = [d[1]*1000 for d in shape_data]
warm_ms  = [d[2]*1000 for d in shape_data]
fig, ax = plt.subplots(figsize=(7, 3.6))
x = np.arange(len(Ns))
ax.bar(x - 0.18, cold_ms, width=0.36, label='cold (first call per shape)', color='#c33')
ax.bar(x + 0.18, warm_ms, width=0.36, label='warm (median, after 2 warmups)', color='#27a')
ax.set_xticks(x); ax.set_xticklabels([f'N={n}' for n in Ns])
ax.set_ylabel('time per N_STEPS=1000 run [ms, log]')
ax.set_yscale('log')
ax.set_title('JAX shape-rigidity: each new N pays one XLA compile')
ax.legend(); ax.grid(alpha=0.3, axis='y')
plt.tight_layout(); plt.show()

**EXTRA CREDIT: inspecting intermediate states**

`lax.scan` runs the whole loop as one compiled program, so there is no Python
loop to `print` from. Inside it `h` is a placeholder (a *tracer*), not real
numbers, so `float(h[i])` fails. To watch the trajectory, have the scan body
return a value each step, and `scan` collects them into an array you get back
at the end.

In [ ]:
state0 = (jnp.asarray(swe_core.bump_ic(N, L=L, h0=H0, amplitude=AMP, sigma=SIG)[0], jnp.float32),
          jnp.zeros(N + 2, jnp.float32))

# A mid-loop h is a tracer, so float(h[i]) inside the scan body raises. The fix
# is to return a per-step diagnostic from the body, which scan stacks into ys.
# TODO: run a scan whose body returns (new_state, per-step total water volume,
# jnp.sum(new_state[0][1:-1])), then block_until_ready and confirm you get one
# sample per step.
...

**Recap.**

- We expressed our computation as a **pure function** of `(h, hu)` and let
  `@jax.jit + lax.scan` fuse 1000 steps into one device program.
- Cold compile cost is significant; warm execution is fast.
- **Shape-rigidity** is the unique limitation surfaced here: changing `N`
  is not free.

Next: PyOMP lets us leverage OpenMP's standard pragma-based API for shared-memory parallelism from Python.